# Optimise the star selection and the match filter on the known objects, per distance modulus bin

For a generic search, the match filter has a fixed age and metallicity (the default isochrone). Here every known
dwarf galaxy and globular cluster of the catalog is searched at its own distance modulus with that isochrone, and the
SNR of its excess (on region against off region, `analysis.gc_dwarf_optimization`) is computed for every combination
of:

- star / galaxy classification (`CLASSIFICATIONS`) and `POINT_LIKE_PROB` cut (`POINT_LIKE_PROB_CUTS`);
- isochrone model (`ISOCHRONE_MODELS`);
- width of the match filter (`WIDTHS`: `color_spread`, `error_multiplier`).

All the SNRs are in one table (`results`); the plots are selections of it. The objects differ a lot in brightness,
so they are compared through their SNR relative to a reference scenario (`REFERENCE`), averaged per distance modulus
bin (`AGGREGATE`). Then the best scenario of each classification, chosen by hand in `BEST`, as a function of the
distance modulus.

Objects without a usable reference (a hole in the catalog at their centre, no background star, no detection:
`MIN_ON_AREA_FRACTION`, `MIN_REFERENCE_OFF_STARS`, `MIN_REFERENCE_SNR`) stay in the per-object panels but are left out
of the bins.

The objects are every LVDB object of the catalog in the bins with the ones made by hand (`HANDMADE_OBJECTS`, e.g.
new discoveries), or those named in `OBJECT_NAMES` (LVDB or made by hand), e.g. a single one.

Every parameter is in the first cells: change them and re-run.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.catalog_preparation import catalog_match_filter_mask
from streameucliddes.analysis.gc_dwarf_optimization import (
    object_field_radius_deg,
    object_radec_box,
    object_regions,
)
from streameucliddes.analysis.on_off_regions import selection_snr
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.isochrone_parameters import default_isochrone_grid
from streameucliddes.catalogs.overlapped_objects import (
    OBJECT_COLUMNS,
    add_in_catalog,
    find_object,
    make_object,
    objects_near_distance_modulus,
    simplified_name,
)
from streameucliddes.plotting.plot_on_off_regions import plot_selection_comparison
from streameucliddes.plotting.plot_style import escape_text

## 1. Parameters

In [ ]:
CATALOG = "euclid_dr1"
OBJECT_TYPES = ("dwarfs", "gc")
DM_BIN_EDGES = np.arange(15.0, 21.01, 1.0)  # distance modulus bins of the objects
EXCLUDED_OBJECTS = []  # names, e.g. ["Fornax"] (large field, slow to read)

# Objects made by hand (e.g. new discoveries, not in the LVDB): make_object(name, ra, dec, half-light radius in arcmin,
# ...), unknown values NaN (age: the default one)
HANDMADE_OBJECTS = [
    # Gatto et al. (Euclid DR1), Tables 1 and 2: dm 16.5 +- 0.3, [Fe/H] ~ -2.2, M_V = +2.0, so V = M_V + dm
    make_object("Ursa Minor II", 196.1542, 75.5724, 2.25, ellipticity=0.20, position_angle=162.0,
                distance_modulus=16.5, feh=-2.2, apparent_magnitude_v=2.0 + 16.5),
]
# Objects studied: None, every LVDB object of the catalog in the bins and the hand-made ones in the bins; or a list of
# names, LVDB objects (find_object) or hand-made ones, e.g. ["Ursa Minor II"] or ["Ursa Minor II", "Reticulum II"]
OBJECT_NAMES = None

# On and off regions (arguments of object_regions; radii in half-light radii): per type, updated per object when it
# needs its own (e.g. bright globular clusters, whose core the catalog misses; an object under the mask of a galaxy)
REGION_OPTIONS = {
    "dwarfs": {"on_radii": (0.0, 2.0), "off_radii": (5.0, 10.0)},
    "gc": {"on_radii": (0.0, 4.0), "off_radii": (10.0, 20.0)},
}
REGION_OPTIONS_BY_NAME = {
    "NGC 1851": {"on_radii": (4.0, 15.0), "off_radii": (30.0, 40.0)},
    "NGC 1261": {"on_radii": (4.0, 15.0), "off_radii": (30.0, 40.0)},
    "Horologium I": {"masked_object_types": ("dwarfs", "gc")},  # its centre is in the mask of the galaxy ESO 154-023
}

# Star / galaxy classifications: the cuts read from the catalog, with the loosest POINT_LIKE_PROB threshold so that
# POINT_LIKE_PROB_CUTS can be applied on top
MIN_POINT_LIKE_PROB = 0.05
CLASSIFICATIONS = {
    "morphology": default_star_cuts(CATALOG, "morphology_cuts", min_point_like_prob=MIN_POINT_LIKE_PROB),
    "gatto": default_star_cuts(CATALOG, "gatto_stars_dr1"),  # POINT_LIKE_PROB >= 0.05 and the colour box
}
POINT_LIKE_PROB_CUTS = np.round(np.arange(MIN_POINT_LIKE_PROB, 0.951, 0.05), 2)  # stars kept: POINT_LIKE_PROB >= cut

# Fixed isochrone (the default age and [Fe/H]), in each model
ISOCHRONE_MODELS = ("Marigo2017", "Dotter2016")
ISOCHRONES = {isochrone_model: default_isochrone_grid(isochrone_model=isochrone_model)[0]
              for isochrone_model in ISOCHRONE_MODELS}

MATCH_FILTER = {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]}
# Width of the match filter: error_multiplier times the photometric error plus color_spread, each (blue side, red side);
# absent: the streamobs defaults, (2, 2) and (0.05, 0.05) mag
WIDTHS = {
    "default": {},
    "color_spread (0.05, 0.1)": {"color_spread": (0.05, 0.1)},  # DES 2018 (Shipp et al.)
    "color_spread (0.1, 0.1)": {"color_spread": (0.1, 0.1)},
    "color_spread (0.1, 0.15)": {"color_spread": (0.1, 0.15)},
    "error_multiplier (1, 1)": {"error_multiplier": (1.0, 1.0)},
    "error_multiplier (3, 3)": {"error_multiplier": (3.0, 3.0)},
    "color_spread (0.1, 0.1) + error_multiplier (3, 3)": {"color_spread": (0.1, 0.1), "error_multiplier": (3.0, 3.0)},
}

# Reference scenario of the relative SNR: the default selection of the searches
REFERENCE = {"classification": "morphology", "isochrone_model": "Marigo2017", "width": "default", "point_like_prob_cut": 0.5}
AGGREGATE = "mean"  # of the relative SNRs of the objects of a bin (finite values only): "mean" or "median"
# Objects used in the bins (the others are still in the per-object panels): enough of the on region in the catalog
# (e.g. no hole at the centre), background stars in the off region, and a detection, in the reference scenario
MIN_ON_AREA_FRACTION = 0.5
MIN_REFERENCE_OFF_STARS = 5
MIN_REFERENCE_SNR = 5.0

# Plot styles
MAX_PANELS_PER_ROW = 4
CLASSIFICATION_COLORS = {"morphology": "C0", "gatto": "C1"}
ISOCHRONE_MODEL_MARKERS = {"Marigo2017": "o", "Dotter2016": "^"}
ISOCHRONE_MODEL_LINESTYLES = {"Marigo2017": "-", "Dotter2016": ":"}
print(ISOCHRONES)

## 2. Objects of the catalog, per distance modulus bin

`OBJECT_NAMES` None: the LVDB objects and the hand-made ones in the bins; else the objects named. Objects without
any source of the catalog at their centre, or outside the bins, are left out.

In [ ]:
handmade = pd.DataFrame(HANDMADE_OBJECTS, columns=["type", *OBJECT_COLUMNS])
if OBJECT_NAMES is None:
    centre, half_width = (DM_BIN_EDGES[0] + DM_BIN_EDGES[-1]) / 2, (DM_BIN_EDGES[-1] - DM_BIN_EDGES[0]) / 2
    lvdb_objects = objects_near_distance_modulus(centre, half_width, object_types=OBJECT_TYPES)
    objects = pd.concat([lvdb_objects, handmade], ignore_index=True)
else:
    handmade_by_name = {simplified_name(row["name"]): row for _, row in handmade.iterrows()}
    objects = pd.DataFrame([handmade_by_name[simplified_name(name)] if simplified_name(name) in handmade_by_name
                            else find_object(name) for name in OBJECT_NAMES])
objects = add_in_catalog(objects, CATALOG)
objects["dm_bin"] = pd.cut(objects["distance_modulus"], DM_BIN_EDGES, right=False)
left_out = objects[~objects["in_catalog"] | objects["dm_bin"].isna()]
if not left_out.empty:
    print("left out (no source of the catalog at the centre, or outside the bins):", left_out["name"].tolist())
objects = objects[objects["in_catalog"] & objects["dm_bin"].notna() & ~objects["name"].isin(EXCLUDED_OBJECTS)]
objects = objects.sort_values("distance_modulus").reset_index(drop=True)
objects["dm_bin_centre"] = objects["dm_bin"].map(lambda interval: interval.mid).astype(float)
objects[["type", "name", "distance_modulus", "dm_bin", "radius_arcmin", "ellipticity", "feh", "apparent_magnitude_v",
         "n_sources_at_centre"]]

## 3. Stars and regions of each object, for each classification

Read once: re-running the next cell only reads the objects not read yet (empty `fields` to read them again).

In [ ]:
fields = {}  # name: {classification: (stars, regions)}

In [ ]:
def region_options(obj):
    """Arguments of object_regions for an object: those of its type, updated with its own."""
    return {**REGION_OPTIONS[obj["type"]], **REGION_OPTIONS_BY_NAME.get(obj["name"], {})}


for _, obj in objects.iterrows():
    if obj["name"] in fields:
        continue
    options = region_options(obj)
    ra_range, dec_range = object_radec_box(obj, object_field_radius_deg(obj, options["off_radii"]))
    fields[obj["name"]] = {}
    for classification, cuts in CLASSIFICATIONS.items():
        stars = get_euclid_stars(CATALOG, cuts=cuts, ra_range=ra_range, dec_range=dec_range, verbose=False)
        regions = object_regions(stars, obj, **options)
        fields[obj["name"]][classification] = (stars, regions)
        print(f"{obj['name']}, {classification}: {len(stars):,} stars, on {regions['on'].sum():,}, "
              f"off {regions['off'].sum():,}")

## 4. SNR of every scenario

One row per object, classification, isochrone model, width and `POINT_LIKE_PROB` cut; each object is searched at its
own distance modulus. `relative_snr`: the SNR over that of the `REFERENCE` scenario for the same object.

In [ ]:
rows = []
for _, obj in objects.iterrows():
    for classification, (stars, regions) in fields[obj["name"]].items():
        point_like_prob = stars["POINT_LIKE_PROB"].to_numpy()
        for (isochrone_model, isochrone), (width_name, width) in itertools.product(ISOCHRONES.items(), WIDTHS.items()):
            in_match_filter = catalog_match_filter_mask(stars, obj["distance_modulus"], isochrone, {**MATCH_FILTER, **width})
            for cut in POINT_LIKE_PROB_CUTS:
                snr = selection_snr(regions, in_match_filter & (point_like_prob >= cut))
                rows.append({"name": obj["name"], "type": obj["type"], "distance_modulus": obj["distance_modulus"],
                             "dm_bin": obj["dm_bin"], "dm_bin_centre": obj["dm_bin_centre"],
                             "on_area_fraction": regions["on_area_fraction"],
                             "classification": classification, "isochrone_model": isochrone_model, "width": width_name,
                             "point_like_prob_cut": cut, **snr})
results = pd.DataFrame(rows)

reference = results[np.logical_and.reduce([results[key] == value for key, value in REFERENCE.items()])].set_index("name")
reference_snr = reference["snr"]
results["relative_snr"] = results["snr"] / results["name"].map(reference_snr)

# Objects used in the bins
usable = ((reference["on_area_fraction"] >= MIN_ON_AREA_FRACTION) & (reference["n_off"] >= MIN_REFERENCE_OFF_STARS)
          & np.isfinite(reference["snr"]) & (reference["snr"] >= MIN_REFERENCE_SNR))
results["usable"] = results["name"].map(usable)
print(f"{len(results):,} scenarios; not used in the bins: {list(usable.index[~usable])}")
reference[["dm_bin", "on_area_fraction", "n_on", "n_off", "snr"]].assign(usable=usable).round(2)

In [ ]:
def select(table, **conditions):
    """Rows of a results table matching every condition (column=value)."""
    keep = np.logical_and.reduce([table[key] == value for key, value in conditions.items()]) if conditions else True
    return table[keep]


def aggregate(values):
    """Relative SNRs of the objects of a bin as one value (AGGREGATE of the finite ones; NaN if none)."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return {"mean": np.mean, "median": np.median}[AGGREGATE](values) if values.size else np.nan


used = results[results["usable"]]  # the rows of the objects used in the bins
def panel_grid(n_panels, panel_width, panel_height, **subplots_kwargs):
    """A figure of n_panels panels, MAX_PANELS_PER_ROW per row: the figure and the list of the panels."""
    n_columns = min(n_panels, MAX_PANELS_PER_ROW)
    n_rows = int(np.ceil(n_panels / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(panel_width * n_columns, panel_height * n_rows),
                             layout="constrained", squeeze=False, **subplots_kwargs)
    axes = axes.ravel()
    for ax in axes[n_panels:]:
        ax.set_visible(False)
    return fig, list(axes[:n_panels])


POINT_LIKE_PROB_LABEL = escape_text("POINT_LIKE_PROB cut")
dm_bins = [interval for interval in objects["dm_bin"].cat.categories if (objects["dm_bin"] == interval).any()]
print("bins with objects:", [str(interval) for interval in dm_bins])
print("objects used per bin:", used.groupby("dm_bin", observed=True)["name"].nunique().to_dict())

## 5. SNR as a function of the `POINT_LIKE_PROB` cut, per distance modulus bin

One panel per object (SNR), and the last one the relative SNR of the bin (`AGGREGATE` over its objects). Lines:
classification (colour) and isochrone model (line style), for the width `POINT_LIKE_PROB_SCAN_WIDTH`.

In [ ]:
POINT_LIKE_PROB_SCAN_WIDTH = "default"

for dm_bin in dm_bins:
    in_bin = select(results, dm_bin=dm_bin, width=POINT_LIKE_PROB_SCAN_WIDTH)
    names = list(dict.fromkeys(in_bin["name"]))
    fig, axes = panel_grid(len(names) + 1, 3.2, 3.0)
    for (classification, color), (isochrone_model, linestyle) in itertools.product(
            CLASSIFICATION_COLORS.items(), ISOCHRONE_MODEL_LINESTYLES.items()):
        scenario = select(in_bin, classification=classification, isochrone_model=isochrone_model)
        style = {"color": color, "ls": linestyle, "marker": ".", "label": f"{classification}, {isochrone_model}"}
        for ax, name in zip(axes, names):
            curve = select(scenario, name=name)
            ax.plot(curve["point_like_prob_cut"], curve["snr"], **style)
        summary = scenario[scenario["usable"]].groupby("point_like_prob_cut")["relative_snr"].apply(aggregate)
        axes[-1].plot(summary.index, summary.to_numpy(), **style)
    for ax, name in zip(axes, names):
        ax.set(title=f"{name} (dm {select(objects, name=name)['distance_modulus'].iloc[0]:g})", ylabel="SNR")
    axes[-1].axhline(1.0, color="0.6", lw=0.8)
    n_used = select(used, dm_bin=dm_bin)["name"].nunique()
    axes[-1].set(title=f"dm {dm_bin}: {AGGREGATE} of {n_used}", ylabel="SNR / reference SNR")
    for ax in axes:
        ax.set_xlabel(POINT_LIKE_PROB_LABEL)
    axes[-1].legend(fontsize="x-small")
    plt.show()

## 6. SNR of the match filter widths and isochrone models, per distance modulus bin

At the `POINT_LIKE_PROB` cut of each classification (`SCENARIO_POINT_LIKE_PROB_CUTS`). One panel per object (SNR) and
the last one the relative SNR of the bin; points: classification (colour) and isochrone model (marker).

In [ ]:
SCENARIO_POINT_LIKE_PROB_CUTS = {"morphology": 0.5, "gatto": 0.55}

for dm_bin in dm_bins:
    in_bin = select(results, dm_bin=dm_bin)
    names = list(dict.fromkeys(in_bin["name"]))
    fig, axes = panel_grid(len(names) + 1, 3.6, 0.8 + 0.3 * len(WIDTHS), sharey=True)
    for (classification, color), (isochrone_model, marker) in itertools.product(
            CLASSIFICATION_COLORS.items(), ISOCHRONE_MODEL_MARKERS.items()):
        scenario = select(in_bin, classification=classification, isochrone_model=isochrone_model,
                          point_like_prob_cut=SCENARIO_POINT_LIKE_PROB_CUTS[classification])
        style = {"marker": marker, "color": color, "label": f"{classification}, {isochrone_model}"}
        for ax, name in zip(axes, names):
            plot_selection_comparison(select(scenario, name=name).set_index("width").loc[list(WIDTHS)], ax=ax, **style)
        summary = (scenario[scenario["usable"]].groupby("width")["relative_snr"].apply(aggregate)
                   .reindex(list(WIDTHS)).rename("snr").to_frame())
        plot_selection_comparison(summary, ax=axes[-1], **style)
    for ax, name in zip(axes, names):
        ax.set(title=name, xlabel="SNR")
    axes[-1].axvline(1.0, color="0.6", lw=0.8)
    axes[-1].set(title=f"dm {dm_bin}: {AGGREGATE} of {select(used, dm_bin=dm_bin)['name'].nunique()}",
                 xlabel="SNR / reference SNR")
    axes[-1].legend(fontsize="x-small", loc="upper left", bbox_to_anchor=(1.0, 1.0))
    plt.show()

## 7. Best scenario of each distance modulus bin

The scenario with the highest relative SNR (`AGGREGATE` over the objects of the bin), for each classification:
a guide to choose `BEST` below.

In [ ]:
scenario_columns = ["isochrone_model", "width", "point_like_prob_cut"]
per_bin = (used.groupby(["dm_bin", "classification", *scenario_columns], observed=True)["relative_snr"]
           .apply(aggregate).rename("relative_snr").reset_index())
best_per_bin = per_bin.loc[per_bin.groupby(["dm_bin", "classification"], observed=True)["relative_snr"].idxmax()]
best_per_bin.round(3)

In [ ]:
# Over all the bins: the scenarios with the highest mean relative SNR (each bin counts once)
overall = per_bin.groupby(["classification", *scenario_columns])["relative_snr"].mean().rename("mean_over_bins")
overall.sort_values(ascending=False).groupby("classification").head(5).round(3)

## 8. Best scenario of each classification, as a function of the distance modulus

`BEST`: one scenario per label (classification, isochrone model, width, `POINT_LIKE_PROB` cut), chosen from the plots
and tables above. Left: SNR of each object at its distance modulus; middle: its SNR relative to the
reference (open markers: objects not used in the bins); right: relative SNR per bin.

In [ ]:
BEST = {
    "reference (morphology, default)": {"classification": "morphology", "isochrone_model": "Marigo2017",
                                        "width": "default", "point_like_prob_cut": 0.5},
    "morphology, best": {"classification": "morphology", "isochrone_model": "Marigo2017",
                         "width": "color_spread (0.1, 0.1)", "point_like_prob_cut": 0.5},
    "gatto, best": {"classification": "gatto", "isochrone_model": "Marigo2017",
                    "width": "color_spread (0.1, 0.1)", "point_like_prob_cut": 0.55},
}
BEST_STYLES = {"reference (morphology, default)": {"color": "0.4", "marker": "s"},
               "morphology, best": {"color": CLASSIFICATION_COLORS["morphology"], "marker": "o"},
               "gatto, best": {"color": CLASSIFICATION_COLORS["gatto"], "marker": "^"}}

fig, (ax_objects, ax_relative, ax_bins) = plt.subplots(1, 3, figsize=(16.0, 4.0), layout="constrained")
for label, scenario in BEST.items():
    chosen = select(results, **scenario)
    ax_objects.plot(chosen["distance_modulus"], chosen["snr"], ls="none", label=label, **BEST_STYLES[label])
    for is_usable, fill in ((True, BEST_STYLES[label]["color"]), (False, "none")):  # open: not used in the bins
        objects_chosen = chosen[chosen["usable"] == is_usable]
        ax_relative.plot(objects_chosen["distance_modulus"], objects_chosen["relative_snr"], ls="none",
                         markerfacecolor=fill, label=label if is_usable else None, **BEST_STYLES[label])
    summary = chosen[chosen["usable"]].groupby("dm_bin_centre")["relative_snr"].apply(aggregate)
    ax_bins.plot(summary.index, summary.to_numpy(), label=label, **BEST_STYLES[label])

# Names (in grey: not used in the bins), at the reference SNR and at the highest finite relative SNR of BEST
best_rows = pd.concat([select(results, **scenario) for scenario in BEST.values()])
highest_relative = best_rows[np.isfinite(best_rows["relative_snr"])].groupby("name")["relative_snr"].max()
for _, obj in objects.iterrows():
    text = {"fontsize": "x-small", "xytext": (3, 3), "textcoords": "offset points",
            "color": "k" if usable[obj["name"]] else "0.6"}
    ax_objects.annotate(obj["name"], (obj["distance_modulus"], reference_snr[obj["name"]]), **text)
    if obj["name"] in highest_relative:
        ax_relative.annotate(obj["name"], (obj["distance_modulus"], highest_relative[obj["name"]]), **text)
ax_objects.set(yscale="log", xlabel="distance modulus", ylabel="SNR", title="Objects")
ax_relative.axhline(1.0, color="0.6", lw=0.8)
ax_relative.set(xlabel="distance modulus", ylabel="SNR / reference SNR", title="Objects, relative to the reference")
ax_bins.axhline(1.0, color="0.6", lw=0.8)
ax_bins.set(xlabel="distance modulus bin", ylabel="SNR / reference SNR", title=f"{AGGREGATE} per bin")
ax_bins.set_xticks(DM_BIN_EDGES[:-1] + np.diff(DM_BIN_EDGES) / 2)
for ax in (ax_relative, ax_bins):
    ax.set_xlim(DM_BIN_EDGES[0], DM_BIN_EDGES[-1])
ax_bins.legend(fontsize="small")
plt.show()